# Árvore de decisão

## Autores

| Nome | nUSP |
| :--- | :--- |
| Débora da Silva Morais | 15615790 |
| Guilherme de Abreu Barreto | 12543033 |

## Introdução

Este notebook trata-se de uma continuidade ao exercício apresentado na semana anterior. Após explorarmos as propriedades dos datasets descritos no artigo de  Zanfei et al., [Novel approach for burst detection in water distribution systems based on graph neural networks](https://www.sciencedirect.com/science/article/abs/pii/S2210670722004073) partimos para uma tentativa para classificação da presença ou não de vazamentos no sistema com o uso da técnica [_Árvore de Decisão_](https://scikit-learn.org/stable/modules/tree.html). A seguir, explicamos as etapas para execução desta análise.

## Carregamento de dependências

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    auc,
)
import matplotlib.pyplot as plt
import seaborn as sns

## Constantes

A seguir são definidos os seguintes atributos:

- `SEED`: valor inteiro utilizado para cálculo de números pseudo-aleatórios. Assegura a reprodutibilidade do notebook em procedimentos probabilísticos como a geração pseudo-aleatória de divisões de treino e validação.

- `DATASETS_FOLDER` e `DATASETS`: Referem-se ao diretório e arquivos onde os dados de referência para este estudo estão localizados.

In [2]:
SEED = 42
DATASETS_FOLDER = '../generated-datasets-for-burst-detection-in-water-distribution-systems/'
DATASETS = [f'{DATASETS_FOLDER}WDts{i}/WDts{i}_dataset.csv' for i in range(1, 5)]

## Carregamento dos dados

A função abaixo carrega os dados a partir dos arquivos de referência, eliminando quaisquer registros em que a variável alvo não esteja presente e atribuindo um valor médio a quaisquer valores faltantes observados nos parâmetros.

In [3]:
def load_data(
    csv_path: str, features: list[str], target: str
) -> tuple[pd.DataFrame, ...]:
    df = pd.read_csv(
        csv_path,
        dtype={target: bool},
        index_col=False,
        sep=';',
    )
    df = df[df[target].notna()]
    df.info()
    return df[features].fillna(df[features].mean()), df[target]

## Dividir conjuntos de treino e teste

Abaixo dividimos o conjunto de registros utilizados para o treino (4/5 do total) do modelo e seu posterior teste (1/5 restante). 

In [4]:
def split_train_test(
    X: pd.DataFrame, y: pd.DataFrame
) -> tuple[pd.DataFrame, ...]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    print(f"Tamanho do conjunto de treino: {X_train.shape}")
    print(f"Tamanho do conjunto de teste: {X_test.shape}")
    return X_train, X_test, y_train, y_test

## Criar o modelo classificador e encontrar os hiperparâmetros ótimos para este

Conforme exemplificado na última aula por Rubens, et al., utilizamos o procedimento de GridSearch com 5-fold cross-validation no conjunto de treino para buscar a combinação ótima de hiperparâmetros do modelo. São testados os seguintes parâmetros:



In [5]:
def tune_hyper_parameters(
    X_train: pd.DataFrame, y_train: pd.DataFrame
) -> tuple[GridSearchCV, DecisionTreeClassifier]:
    dt = DecisionTreeClassifier(random_state=SEED)
    param_grid = {
        'criterion': ["gini", "entropy", "log_loss"],
        'max_depth': range(1, 21),
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    gs = GridSearchCV(
        dt, param_grid, cv=5, scoring='roc_auc', n_jobs=-1
    )
    gs.fit(X_train, y_train)
    best_dt = gs.best_estimator_
    return gs, best_dt.fit(X_train, y_train)

## Avaliar predição do modelo sobre o conjunto de teste

In [6]:
def evaluate_results(
    gs: GridSearchCV,
    dt: DecisionTreeClassifier,
    X_test: pd.DataFrame,
    y_test: pd.DataFrame,
    testset_name: str,
    features: list[str]
) -> None:
    y_pred = dt.predict(X_test)

    # Probabilities for class 1 (burst)
    y_pred_proba = dt.predict_proba(X_test)[:, 1]
    test_roc_auc = roc_auc_score(y_test, y_pred_proba)
    test_accuracy = accuracy_score(y_test, y_pred)

    print("\nRelatório de classificação:")
    print(classification_report(y_test, y_pred))

    # Print Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    cm_percentage = cm.astype('float') /cm.sum(axis=1)[:, np.newaxis] * 100
    plt.figure(figsize=(8,6))
    labels = ["Verdadeiro", "Falso"]
    sns.heatmap(
        cm_percentage,
        annot=True,
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels
    )
    plt.xlabel('Previsão')
    plt.ylabel('Realidade')
    plt.title(f"Matrix de confusão, conjunto de teste {testset_name}")
    plt.show

    # Plot ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(
        fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})'
    )
    plt.plot(
        [0, 1],
        [0, 1],
        color='navy',
        lw=2,
        linestyle='--',
        label='Classificador aleatório'
    )
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taxa de falsos positivos')
    plt.ylabel('Taxa de positivos verdadeiros')
    plt.title(f'Curva ROC, conjunto de teste {testset_name}')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

    # Plot decision tree
    ax = plt.subplots(figsize=(12,12))[1]
    plot_tree(dt, feature_names=features, ax=ax)

    # Plot feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance, x='importância', y='parâmetro')
    plt.title(
        f'Importância do parâmetro na árvore de decisão, conjunto de teste {testset_name}'
    )
    plt.tight_layout()
    plt.show()

    # Final model performance summary
    print("\n" + "=" * 60)
    print("SUMÀRIO FINAL DO MODELO")
    print("="*60)
    print(
        f"Melhor critério de separação: {gs.best_params_['criterion']}"
    )
    print(
        f"Melhor profundidade máxima: {gs.best_params_['max_depth']}"
    )
    print(
        f"Melhor número mínimo de amostras por divisão: {gs.best_params_['min_samples_split']}"
    )
    print(
        f"Melhor número mínimo de amostras por folha: {gs.best_params_['min_samples_leaf']}"
    )
    print(
        f"Melhor pontuação ROC AUC na validação cruzada: {gs.best_score_:.4f}"
    )
    print(
        f"Pontuação ROC AUC no conjunto de teste: {test_roc_auc:.4f}"
    )
    print(
        f"Acurácia no conjunto de teste: {test_accuracy:.4f}"
    )
    print(
        f"Número de parâmetros: {len(features)}"
    )
    print(
        f"Parâmetro mais impactante: {feature_importance.iloc[0]['feature']} "
        f"(imporância: {feature_importance.iloc[0]['importance']:.4f})"
    )

## Resultados

In [7]:
def decision_tree_pipeline(csv_path:str) -> None:
    target = 'burst'
    features = [
        f'flow_meter{i}' for i in range(1, 5)
    ] + [
        f'{i}_press' for i in [3, 12, 36, 50, 60, 84, 93, 112, 138, 139]
    ]
    X, y = load_data(csv_path, features, target)
    X_train, X_test, y_train, y_test = split_train_test(X, y)
    grid_search, decision_tree = tune_hyper_parameters(X_train, y_train)
    evaluate_results(
        grid_search,
        decision_tree,
        X_test,
        y_test,
        csv_path.rsplit('/', 1)[-1],
        features
    )

### Conjunto WDts1

In [ ]:
decision_tree_pipeline(DATASETS[0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35040 entries, 0 to 35039
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   index        35040 non-null  object 
 1   burst        35040 non-null  bool   
 2   flow_meter1  35040 non-null  float64
 3   flow_meter2  35040 non-null  float64
 4   flow_meter3  35040 non-null  float64
 5   flow_meter4  35040 non-null  float64
 6   3_press      35040 non-null  float64
 7   12_press     35040 non-null  float64
 8   36_press     35040 non-null  float64
 9   50_press     35040 non-null  float64
 10  60_press     35040 non-null  float64
 11  84_press     35040 non-null  float64
 12  93_press     35040 non-null  float64
 13  112_press    35040 non-null  float64
 14  138_press    35040 non-null  float64
 15  139_press    35040 non-null  float64
dtypes: bool(1), float64(14), object(1)
memory usage: 4.0+ MB
Tamanho do conjunto de treino: (28032, 14)
Tamanho do conjunto de teste: 

### Conjunto WDts2

In [ ]:
decision_tree_pipeline(DATASETS[1])

### Conjunto WDts3

In [ ]:
decision_tree_pipeline(DATASETS[2])

### Conjunto WDts4

In [ ]:
decision_tree_pipeline(DATASETS[3])